# DQN Space Invaders — train on Colab, resume from Drive

**One-time setup**
1. Upload the whole `deep_q_learning` folder to the top of My Drive, so you end up with
   `MyDrive/deep_q_learning/src/...` (only `src/` and `play.py` are actually needed).
2. In Colab: *Runtime → Change runtime type → T4 GPU*.

**Every session:** *Runtime → Run all*. It mounts Drive, resumes from the newest checkpoint in
`MyDrive/deep_q_learning/checkpoints/` (or starts fresh), and trains until you press stop or Colab disconnects.
The chart under the training cell refreshes live; a copy is also saved to `checkpoints/progress.png`
so you can peek from the Drive app.

To watch the agent: stop the training cell, then run the last cell.

In [ ]:
# --- Settings ---------------------------------------------------------------
CODE_DIR = "/content/drive/MyDrive/deep_q_learning"   # project folder in Drive (contains src/ and play.py)
CHECKPOINT_DIR = f"{CODE_DIR}/checkpoints"            # weights + progress log are kept here

CHECKPOINT_EVERY = 100   # episodes between checkpoints (the project's default)
PLOT_EVERY = 5           # episodes between chart refreshes
MAX_EPISODES = None      # None = run until you stop it or Colab disconnects

# Overrides for the defaults in src/train.py
EPSILON_DECAY = 0.9985   # per episode: epsilon hits 0.1 after ~1,500 episodes (~1M steps), like the DQN paper (src default 0.9999 = ~23,000 episodes)
DISCOUNT = 0.99          # standard DQN value (src default 0.9 only looks ~10 steps ahead)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q "gymnasium[atari]>=1.0" "ale-py>=0.10" "imageio[ffmpeg]" tqdm

### Import the project
The code was written against gymnasium 0.29, whose Atari dependency (`ale-py 0.8`) can't be installed
on Colab's Python. The `*V0` wrappers it imports from `gymnasium.experimental.wrappers` became the regular
`gymnasium.wrappers` in 1.0, so this cell just aliases them under the old import path — the project code is
used unmodified.

In [ ]:
import functools, os, sys, types
import gymnasium as gym
import ale_py
import torch
from gymnasium import wrappers

gym.register_envs(ale_py)  # registers "ALE/SpaceInvaders-v5"

legacy = types.ModuleType("gymnasium.experimental.wrappers")
legacy.AtariPreprocessingV0 = wrappers.AtariPreprocessing
legacy.FrameStackObservationV0 = functools.partial(wrappers.FrameStackObservation, padding_type="zero")
legacy.ClipRewardV0 = wrappers.ClipReward
sys.modules["gymnasium.experimental"] = types.ModuleType("gymnasium.experimental")
sys.modules["gymnasium.experimental.wrappers"] = legacy

sys.path.insert(0, CODE_DIR)
from src.train import DQN
import play

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only!)")

### Create the agent and resume
Loads the newest `.pt` in the checkpoint folder (falls back to an older one if the newest is corrupt, e.g.
Colab died mid-save). It also hooks into `learn()` / `episode()` to record the loss per episode to
`checkpoints/progress.csv`, which is what the chart is drawn from, so the history survives restarts.

In [ ]:
import csv, glob, io, math, time
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
PROGRESS_FILE = f"{CHECKPOINT_DIR}/progress.csv"
LOG_FILE = f"{CHECKPOINT_DIR}/train_log.txt"
FIELDS = ["time", "total_steps", "episode_steps", "score", "epsilon", "loss"]

# TensorBoard logs a scalar every step, which would bloat Drive, so it stays local to this session.
dqn = DQN(checkpoint_dir=CHECKPOINT_DIR, log_dir="/content/tensorboard_logs")
dqn.epsilon_decay, dqn.discount = EPSILON_DECAY, DISCOUNT

for path in sorted(glob.glob(f"{CHECKPOINT_DIR}/*.pt"), key=os.path.getmtime, reverse=True):
    try:
        dqn.load_checkpoint(os.path.basename(path))
        print(f"Resumed from {os.path.basename(path)}: {dqn.total_steps:,} steps, epsilon={dqn.epsilon:.4f}")
        break
    except Exception as e:
        print(f"Could not load {os.path.basename(path)} ({e}), trying an older one")
else:
    print("No checkpoint found, starting from scratch")

# Drop progress rows logged after the checkpoint we resumed from (work lost when Colab disconnected).
history = pd.read_csv(PROGRESS_FILE) if os.path.exists(PROGRESS_FILE) else pd.DataFrame(columns=FIELDS)
history = history[history.total_steps <= dqn.total_steps]
history.to_csv(PROGRESS_FILE, index=False)
history = history.to_dict("records")
print(f"{len(history)} episodes of history")

chart = None  # display handle, created in the training cell
session_start = (time.time(), dqn.total_steps)
episode_losses = []


def draw_chart():
    df = pd.DataFrame(history)
    window = max(1, min(50, len(df) // 10))
    fig, (ax_loss, ax_reward) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

    ax_loss.plot(df.total_steps, df.loss, alpha=0.3, lw=0.8, label="episode mean")
    ax_loss.plot(df.total_steps, df.loss.rolling(window, min_periods=1).mean(), lw=2, label=f"{window}-episode avg")
    ax_loss.set_yscale("log")
    ax_loss.set_ylabel("loss (Huber)")
    ax_loss.legend(loc="upper right")

    ax_reward.plot(df.total_steps, df.score, alpha=0.3, lw=0.8, label="episode")
    ax_reward.plot(df.total_steps, df.score.rolling(window, min_periods=1).mean(), lw=2, label=f"{window}-episode avg")
    ax_reward.set_ylabel("game score")
    ax_reward.set_xlabel("training steps")
    ax_reward.legend(loc="upper left")

    elapsed, steps = time.time() - session_start[0], dqn.total_steps - session_start[1]
    last = history[-1]
    fig.suptitle(f"{len(df):,} episodes · {last['total_steps']:,} steps · epsilon {last['epsilon']:.3f} · "
                 f"{steps / max(elapsed, 1):.0f} steps/s · updated {datetime.now():%H:%M:%S}")
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=80)
    plt.close(fig)
    with open(f"{CHECKPOINT_DIR}/progress.png", "wb") as f:
        f.write(buf.getvalue())
    if chart is not None:
        chart.update(Image(data=buf.getvalue()))


original_learn, original_episode = dqn.learn, dqn.episode


def learn_and_record(step_i):
    loss = original_learn(step_i)
    episode_losses.append(loss.item())
    return loss


def episode_and_record():
    episode_losses.clear()
    rewards, actions, score = original_episode()
    row = {
        "time": datetime.now().isoformat(timespec="seconds"),
        "total_steps": dqn.total_steps,
        "episode_steps": len(rewards),
        "score": score,
        "epsilon": dqn.epsilon,
        "loss": sum(episode_losses) / len(episode_losses) if episode_losses else math.nan,
    }
    history.append(row)
    with open(PROGRESS_FILE, "a", newline="") as f:
        csv.DictWriter(f, FIELDS).writerow(row)
    if len(history) % PLOT_EVERY == 0:
        draw_chart()
    return rewards, actions, score


dqn.learn, dqn.episode = learn_and_record, episode_and_record

### Train
Runs until you press stop (■) or Colab disconnects. Stopping saves `interrupted.pt`; a disconnect loses
at most `CHECKPOINT_EVERY` episodes. The per-episode console output goes to `checkpoints/train_log.txt`
instead of here, so the notebook doesn't grow huge over hours.

*Reading the chart:* DQN loss doesn't fall steadily like supervised loss. What you want is a loss that stays
in a bounded band (not climbing off towards infinity) and a score average that trends up as epsilon drops.
A random agent scores ~150–250; a learning one should pass 500 once epsilon is low.

In [ ]:
import contextlib

chart = display(Markdown(f"Training… first chart after {PLOT_EVERY} episodes."), display_id=True)
if history:
    draw_chart()
try:
    with open(LOG_FILE, "a") as log, contextlib.redirect_stdout(log):
        dqn.train(max_episodes=MAX_EPISODES, checkpoint_every=CHECKPOINT_EVERY)
    dqn.save_checkpoint("latest.pt")
except KeyboardInterrupt:
    dqn.save_checkpoint("interrupted.pt")
    print(f"Stopped. Saved interrupted.pt + latest.pt at {dqn.total_steps:,} steps")

### Watch it play
Runs the project's `play.play_game()` on a checkpoint. Colab has no screen, so the `render_mode="human"`
window is swapped for frame capture and shown as a video below (also saved to Drive).

In [ ]:
PLAY_CHECKPOINT = f"{CHECKPOINT_DIR}/latest.pt"
PLAY_EPISODES = 1

import imageio
from IPython.display import Video


class FrameRecorder(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.frames = []

    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        self.frames.append(self.env.render())
        return result

    def step(self, action):
        result = self.env.step(action)
        self.frames.append(self.env.render())
        return result


recorders = []


def make_with_capture(*args, **kwargs):
    kwargs["render_mode"] = "rgb_array"
    recorders.append(FrameRecorder(gym.make(*args, **kwargs)))
    return recorders[-1]


play.gym = types.SimpleNamespace(make=make_with_capture)  # only affects play.py
play.play_game(PLAY_CHECKPOINT, num_episodes=PLAY_EPISODES, render_delay=0)

video_path = f"{CHECKPOINT_DIR}/play_{datetime.now():%Y%m%d_%H%M%S}.mp4"
imageio.mimsave(video_path, recorders[-1].frames, fps=60, macro_block_size=1)
display(Video(video_path, embed=True, width=480))